# Explore: RAG (Retrieval-Augmented Generation)

This notebook demonstrates how the RAG agent retrieves relevant
documents and uses them to generate advice.

**What you'll learn:**
1. How text embeddings represent meaning as numbers
2. How FAISS finds similar documents
3. How RAG improves LLM answers with context
4. How to measure retrieval quality

**Note:** Steps using LiteLLM for embeddings need an API key.
We provide a fallback using random vectors for learning FAISS mechanics.

In [ ]:
import json
import numpy as np
import faiss
from pathlib import Path

print('All imports successful!')

## Step 1: Load the Knowledge Base

Our knowledge base is a collection of financial advice documents.
Each document has a title, content, and category.

In [ ]:
# Load knowledge base
kb_path = Path('../data/financial_knowledge.json')
if kb_path.exists():
    with open(kb_path) as f:
        documents = json.load(f)
else:
    print('Knowledge base file not found, using inline data')
    documents = [
        {'id': 1, 'title': '50/30/20 Budget Rule', 'content': 'Allocate 50% to needs, 30% to wants, 20% to savings.', 'category': 'budgeting'},
        {'id': 2, 'title': 'Emergency Fund', 'content': 'Save 3-6 months of expenses in a high-yield account.', 'category': 'savings'},
        {'id': 3, 'title': 'Inflation Impact', 'content': 'If inflation exceeds savings interest, money loses purchasing power.', 'category': 'inflation'},
    ]

print(f'Loaded {len(documents)} documents\n')
for doc in documents:
    print(f"  [{doc['category']}] {doc['title']}")

## Step 2: Understanding Embeddings

Embeddings convert text to vectors (lists of numbers).
Similar texts → similar vectors → close in vector space.

We'll use random vectors here to demonstrate FAISS mechanics.
Replace with real embeddings (LiteLLM) for meaningful retrieval.

In [ ]:
# --- Option A: Random vectors (works without API key) ---
USE_REAL_EMBEDDINGS = False  # Set to True if you have an API key

if USE_REAL_EMBEDDINGS:
    import litellm
    import os
    os.environ['OPENAI_API_KEY'] = 'sk-...'  # Your key here
    
    texts = [f"{doc['title']}: {doc['content']}" for doc in documents]
    response = litellm.embedding(model='text-embedding-ada-002', input=texts)
    embeddings = np.array([item['embedding'] for item in response.data], dtype='float32')
    print(f'Real embeddings shape: {embeddings.shape}')
else:
    # Simulate embeddings: each category gets a cluster of similar vectors
    # This demonstrates that FAISS retrieval works based on vector similarity
    np.random.seed(42)
    dim = 128
    
    # Create category centroids (so same-category docs are close)
    category_centroids = {
        'budgeting': np.random.rand(dim).astype('float32'),
        'savings': np.random.rand(dim).astype('float32'),
        'inflation': np.random.rand(dim).astype('float32'),
        'spending': np.random.rand(dim).astype('float32'),
        'debt': np.random.rand(dim).astype('float32'),
        'income': np.random.rand(dim).astype('float32'),
    }
    
    embeddings = []
    for doc in documents:
        centroid = category_centroids.get(doc['category'], np.random.rand(dim))
        # Add noise around the centroid
        vec = centroid + np.random.normal(0, 0.1, dim).astype('float32')
        embeddings.append(vec)
    
    embeddings = np.array(embeddings, dtype='float32')
    print(f'Simulated embeddings shape: {embeddings.shape}')
    print(f'(Using category-clustered random vectors for demo)')

print(f'\nEach document is now a vector of {embeddings.shape[1]} numbers')
print(f'First vector (truncated): [{embeddings[0][0]:.4f}, {embeddings[0][1]:.4f}, {embeddings[0][2]:.4f}, ...]')

## Step 3: Build the FAISS Index

FAISS stores vectors and enables fast nearest-neighbour search.

`IndexFlatL2` uses L2 (Euclidean) distance — the straight-line
distance between two points in vector space. Smaller distance = more similar.

In [ ]:
# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

# Add all document vectors
index.add(embeddings)

print(f'FAISS index created:')
print(f'  Dimension: {dimension}')
print(f'  Vectors stored: {index.ntotal}')
print(f'  Index type: Flat L2 (exact search)')

## Step 4: Search the Index

Let's simulate a user query and find the most relevant documents.

In [ ]:
def search(query_text, top_k=3):
    """Search the FAISS index and return matching documents."""
    if USE_REAL_EMBEDDINGS:
        response = litellm.embedding(model='text-embedding-ada-002', input=[query_text])
        query_vec = np.array([response.data[0]['embedding']], dtype='float32')
    else:
        # For demo: use a vector close to a category centroid
        # In reality, the embedding model would create this from the text
        if 'inflation' in query_text.lower():
            query_vec = category_centroids['inflation'] + np.random.normal(0, 0.05, dim)
        elif 'save' in query_text.lower() or 'saving' in query_text.lower():
            query_vec = category_centroids['savings'] + np.random.normal(0, 0.05, dim)
        elif 'budget' in query_text.lower():
            query_vec = category_centroids['budgeting'] + np.random.normal(0, 0.05, dim)
        elif 'spend' in query_text.lower():
            query_vec = category_centroids['spending'] + np.random.normal(0, 0.05, dim)
        else:
            query_vec = np.random.rand(dim)
        query_vec = query_vec.astype('float32').reshape(1, -1)
    
    # Search!
    distances, indices = index.search(query_vec, top_k)
    
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        doc = documents[idx]
        relevance = round(1 / (1 + dist), 3)  # Convert distance to 0-1 score
        results.append({**doc, 'distance': round(float(dist), 3), 'relevance': relevance})
    
    return results


# Test queries
test_queries = [
    'How do I deal with inflation eating my savings?',
    'What is a good budget rule for beginners?',
    'How to reduce my monthly spending?',
]

for query in test_queries:
    print(f'\nQuery: "{query}"')
    results = search(query, top_k=3)
    for i, r in enumerate(results):
        print(f'  {i+1}. [{r["category"]}] {r["title"]} (relevance: {r["relevance"]})')

## Step 5: The RAG Prompt

Here's how we build the prompt that combines retrieved documents
with the user's question. This is what gets sent to the LLM.

In [ ]:
def build_rag_prompt(question, retrieved_docs, user_context=''):
    """Build the RAG prompt from retrieved documents."""
    
    # System instructions
    system = (
        'You are a certified financial advisor specialising in personal '
        'budgeting for Singapore residents. Give specific, actionable advice. '
        'Base your answer on the provided knowledge base documents.'
    )
    
    # Retrieved context
    context_parts = []
    for doc in retrieved_docs:
        context_parts.append(f"[{doc['title']}]: {doc['content']}")
    context = '\n\n'.join(context_parts)
    
    # User message
    user_msg = (
        f'KNOWLEDGE BASE CONTEXT:\n{context}\n\n'
        f'USER SITUATION:\n{user_context or "Not specified"}\n\n'
        f'QUESTION: {question}'
    )
    
    return system, user_msg


# Demo: build a prompt
question = 'How do I save money when inflation is high?'
docs = search(question, top_k=3)
system, user_msg = build_rag_prompt(
    question, docs, 
    user_context='I earn $5000/month, single, renting in Singapore'
)

print('=== SYSTEM PROMPT ===')
print(system)
print('\n=== USER MESSAGE ===')
print(user_msg)
print('\n=== SOURCES USED ===')
for doc in docs:
    print(f'  - {doc["title"]} (relevance: {doc["relevance"]})')

## Step 6: Distance Visualization

Let's visualize how documents cluster by category in vector space.
We'll use PCA to reduce dimensions to 2D for plotting.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Reduce to 2D for visualization
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))

category_colors = {
    'budgeting': '#2196F3', 'savings': '#4CAF50', 'inflation': '#F44336',
    'spending': '#FF9800', 'debt': '#9C27B0', 'income': '#795548',
}

for i, doc in enumerate(documents):
    color = category_colors.get(doc['category'], 'gray')
    ax.scatter(coords[i, 0], coords[i, 1], c=color, s=100, alpha=0.7)
    ax.annotate(
        doc['title'][:25] + '...' if len(doc['title']) > 25 else doc['title'],
        (coords[i, 0], coords[i, 1]),
        fontsize=8, ha='center', va='bottom',
    )

# Legend
for cat, color in category_colors.items():
    ax.scatter([], [], c=color, label=cat, s=60)
ax.legend(title='Category', loc='best')

ax.set_title('Document Embeddings (PCA 2D Projection)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Documents in the same category should cluster together.')
print('FAISS finds the nearest points to a query vector in this space.')

## Exercises for You

1. **Enable real embeddings**: Set `USE_REAL_EMBEDDINGS = True` with your API key. Compare retrieval quality with simulated vs real embeddings.
2. **Add new documents**: Add 3 documents about topics you care about. Rebuild the index and test retrieval.
3. **Change top_k**: Try k=1 and k=5. How does it affect the RAG prompt quality?
4. **Measure retrieval accuracy**: Create 10 test questions where you know the right document. Run them and count how often FAISS retrieves the correct document in top-3.